In [1]:
## Defining packages
import os
import shutil
import pathlib
from datetime import datetime
from PIL import Image
from PIL.ExifTags import TAGS
from pathlib import Path
import piexif

## Defining Variables
datetime_original_tag = 36867
datetime_digitized_tag = 36868
datetime_tag = 306

In [2]:
source_base = "C:/Users/laram/Pictures/addexifdata - source/"
destination_base = "C:/Users/laram/Pictures/addexifdata - dest/"

In [14]:
def filename_timestamp_splitter(datetime_string, date_delimiter='-', space_delimiter=' ', time_delimiter=''):
    
    print(datetime_string)
    date, time = datetime_string.split(space_delimiter)
    date_split = date.split(date_delimiter)
    
    if time_delimiter:
        time_split = "".join(time.split(time_delimiter))
    else:
        time_split = time
    
    timestamp_string = "".join(date_split) + time_split
    return timestamp_string

In [4]:
def format_exif_timestamp(datetime_string):
    
    year = datetime_string[:4]
    month = datetime_string[4:6]
    day = datetime_string[6:8]
    hour = datetime_string[8:10]
    minute = datetime_string[10:12]
    second = datetime_string[12:]
    
    return f"{year}:{month}:{day} {hour}:{minute}:{second}"

In [5]:
def is_real_date(dt):
    
    #TODO: add delimiter parameters so this is dynamic for potential differences in datetime format
    try:
        datetime.strptime(dt[:19], "%Y-%m-%d %H:%M:%S")
        return True
    except:
        return False

In [6]:
def has_exif_datetime(image_path):
    
    img = Image.open(image_path)
    exif_data = img._getexif()
    
    original_datetime = exif_data.get(datetime_original_tag)
    digitized_datetime = exif_data.get(datetime_digitized_tag)
    img_datetime = exif_data.get(datetime_tag)

    true_datetime = original_datetime or digitized_datetime or img_datetime
    
    if true_datetime:
        return True
    else:
        print("No EXIF DateTime data: " +  image_path)
        return False

# Steps

1. check if filename has valid datetime name (if not = ignore, no way to set exif data)
2. check if there is exif data for main datetime original (if there is = ignore, no reason to add datetime data)
   - but check first that it matches the filename, flag it if it doesnt
3. check if there is exif data for other datetime values (if there is, must do some validation)
   - if there is no exif datetime data at all -> easy = use filename datetime
   - if there is a datetime value -> check that it matches the filename
      - if it does match the filename -> good to set all datetimes to that value
      - if it does not match the filename -> ignore and flag the file as contradictory
4. copy image to destination
6. add all datetime values with the one found in filename (if previous checks have passed)
7. do some validation that the datetime value has changed and maybe some validation that the rest of the exif hasn't been altered? if not overkill?
8. once the file has been moved (check it exists) and there's a datetime original value, delete the original image from the source path

In [15]:
contradictary_files = {}

folder = Path(source_base)
for file in folder.rglob("*"):
    
    # Defining file-specific variables
    file_source_path = str(file)
    filename = Path(file_source_path).name
    img = Image.open(file_source_path)
    exif_data = img._getexif()
    original_datetime = exif_data.get(datetime_original_tag)
    digitized_datetime = exif_data.get(datetime_digitized_tag)
    plain_datetime = exif_data.get(datetime_tag)
    filename_datetime = filename_timestamp_splitter(filename[:17])
    
    
    # Step 1: Ignore if there is no datetime found in the filename
    if not is_real_date(filename_datetime):
        continue

    filename_datetime_formatted = format_exif_timestamp(filename_datetime)
    
    
    # Step 2: Ignore if there is already a datetime found in the EXIF data (but with consolidation checks)
    if has_exif_datetime(image_path):
        
        # Step 2.5: Flag the file if the filename datetime is different from the EXIF Datetime original
        if original_datetime != filename_datetime_formatted:
            contradictary_files[file_source_path] = original_datetime

        continue
        
    # Step 3: Check that other exif datetime values don't differ from the filename datetime
    if digitized_datetime or plain_datetime:
        other_exif_datetime = digitized_datetime or plain_datetime
        if other_exif_datetime != filename_datetime_formatted:
            contradictary_files[file_source_path] = other_exif_datetime
            continue
            
    # Happy that the filename will be the datetime value for the exif data
    final_exif_datetime = filename_datetime_formatted
    
    # Step 4: Copy image to the destination
    file_dest_path = destination_base + filename
    try:
        shutil.copyfile(file_source_path, file_dest_path)
    except:
        print("Error copying file: " + file_path)
        continue

    # Step 5: Add datetime values to exif data if they don't already exist
    exif_dict = piexif.load(img.info.get("exif", b""))
    
    exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal]= datetime_str
    
    if not exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized]:
        exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized] = datetime_str
    
    if not exif_dict["0th"][piexif.ImageIFD.DateTime]:
        exif_dict["0th"][piexif.ImageIFD.DateTime] = datetime_str
        
    exif_bytes = piexif.dump(exif_dict)
    
    img.save(image_path, exif=exif_bytes)
    
    # Step 6: Validation
    if Path(file_dest_path).exists():
        new_img = Image.open(file_dest_path)
        new_exif_data = new_img._getexif()
        if exif_data.get(datetime_original_tag):
            os.remove(file_source_path)
        else:
            print("Datetime data not found in EXIF:", file_dest_path)
            continue
    else:
        print("Copy failed, not deleting:", source)
        continue
    
    

2004-12-09 171148
2018-01-01_120105


ValueError: not enough values to unpack (expected 2, got 1)

## Testing

This section is for testing functionality on individual photos

In [47]:
has_exif_datetime(image_path)

No EXIF data: C:/Users/laram/Pictures/Dropbox Copy - All data/Emma/a.jpg


False

In [82]:
exif_dict = piexif.load(img.info.get("exif", b""))

In [86]:
exif_dict["0th"][piexif.ImageIFD.DateTime]

b'2004:12:09 17:11:48'

In [87]:
image_path = "C:/Users/laram/Pictures/addexifdata - source/2004-12-09 1711480 - Ellens birthday pictures.JPG"
img = Image.open(image_path)
exif_data = img._getexif()

exif_info = {}
for tag, value in exif_data.items():
    tag_name = TAGS.get(tag, tag)
    #print(str(tag_name) + " (" + str(tag) + ") : " + str(value))

In [41]:
original_datetime = exif_data.get(datetime_original_tag)
digitized_datetime = exif_data.get(datetime_digitized_tag)
img_datetime = exif_data.get(datetime_tag)

true_datetime = original_datetime or digitized_datetime or img_datetime
# Use this logic in the renaming script***

In [95]:
# image_path[:19]
img_filename = Path(image_path).name
# img_filename[:19]
dt_string_unformatted = filename_timestamp_splitter(img_filename[:17])
format_exif_timestamp(dt_string_unformatted)

'2004:12:09 17:11:48'

In [42]:
true_datetime

'2004:12:09 17:11:48'

In [46]:
image_path = "C:/Users/laram/Pictures/Dropbox Copy - All data/Emma/a.jpg"
img = Image.open(image_path)
exif_data = img._getexif()

exif_info = {}
for tag, value in exif_data.items():
    tag_name = TAGS.get(tag, tag)
    print(str(tag_name) + " : " + str(value))

ExifOffset : 26
ColorSpace : 1
